# <font color="#418FDE" size="6.5" uppercase>**Unüberwacht lernen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Vergleichen KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture. 
- Wenden IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierte Daten an. 
- Untersuchen semi-supervised Verfahren mit wenigen Labels und dokumentieren Unsicherheit. 


## **1. Clusterverfahren vergleichen**

### **1.1. KMeans im Vergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_01.jpg?v=1784802597" width="250">



>* KMeans gruppiert Daten um ähnliche Zentren
>* Textcluster brauchen sorgfältige inhaltliche Prüfung

>* Schnell und skalierbar, aber annahmenreich
>* Empfindlich gegenüber Skalierung, Ausreißern und Clusterformen

>* Clusterzentren helfen, Gruppen verständlich zu beschreiben
>* KMeans bleibt hart, geometrisch und prüfpflichtig



In [ ]:
#@title Python-Code - KMeans im Vergleich

# Dieses Beispiel vergleicht KMeans mit MiniBatchKMeans.
# Beide Verfahren suchen kompakte Clusterzentren.
# Die Grafik zeigt ähnliche, aber nicht identische Zuordnungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# Wir erzeugen einfache zweidimensionale Daten mit drei Gruppen.
features, true_labels = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=1.2,
    random_state=42,
)

# Skalierung macht Abstände zwischen Merkmalen fairer vergleichbar.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Eine kurze Prüfung verhindert unklare Fehlermeldungen später.
if scaled_features.shape != (450, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# KMeans nutzt alle Datenpunkte in jeder Aktualisierung.
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(scaled_features)

# MiniBatchKMeans nutzt kleine Datenpakete und ist oft schneller.
mini_batch = MiniBatchKMeans(
    n_clusters=3,
    batch_size=60,
    n_init=10,
    random_state=42,
)

mini_batch_labels = mini_batch.fit_predict(scaled_features)

# Der ARI vergleicht Clusterzuordnungen ohne feste Cluster-Namen.
ari_score = adjusted_rand_score(kmeans_labels, mini_batch_labels)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Ähnlichkeit der Zuordnungen, ARI: {ari_score:.3f}")
print(f"KMeans Trägheit: {kmeans.inertia_:.1f}")
print(f"MiniBatchKMeans Trägheit: {mini_batch.inertia_:.1f}")

# Die Grafik zeigt KMeans-Farben und beide Arten von Zentren.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=kmeans_labels,
    cmap="viridis",
    s=28,
    alpha=0.75,
)

ax.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    c="red",
    marker="X",
    s=180,
    label="KMeans-Zentren",
)

ax.scatter(
    mini_batch.cluster_centers_[:, 0],
    mini_batch.cluster_centers_[:, 1],
    c="white",
    edgecolors="black",
    marker="o",
    s=120,
    label="MiniBatch-Zentren",
)

ax.set_title("KMeans und MiniBatchKMeans auf denselben Daten")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend()
plt.show()



### **1.2. MiniBatch und Silhouette**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_02.jpg?v=1784802601" width="250">



>* Mini-Batches beschleunigen Clustering großer Datensätze
>* Etwas weniger genau, dafür ressourcenschonend

>* MiniBatchKMeans skaliert für riesige Textmengen
>* Cluster fachlich prüfen, Ergebnisse können schwanken

>* Silhouette bewertet Passung und Trennung von Clustern
>* Immer mit Fachwissen und Visualisierung prüfen



In [ ]:
#@title Python-Code - MiniBatch und Silhouette

# Wir vergleichen MiniBatchKMeans mit verschiedenen Clusterzahlen.
# Die Silhouette bewertet Trennung und Kompaktheit.
# Das Diagramm zeigt die beste einfache Wahl.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir erzeugen kleine, gut sichtbare Beispieldaten.
features, true_groups = make_blobs(
    n_samples=600, centers=4, cluster_std=1.2, random_state=42
)

# Skalierung macht Abstände zwischen Merkmalen fairer vergleichbar.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Diese Prüfung schützt vor unerwartet falschen Datenformen.
if scaled_features.shape != (600, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Wir testen mehrere mögliche Clusterzahlen systematisch.
cluster_counts = [2, 3, 4, 5, 6]
silhouette_values = []

for cluster_count in cluster_counts:
    model = MiniBatchKMeans(
        n_clusters=cluster_count, batch_size=64, n_init=10, random_state=42
    )
    labels = model.fit_predict(scaled_features)
    score = silhouette_score(scaled_features, labels)
    silhouette_values.append(score)

# Der höchste Silhouette-Wert markiert hier die beste getestete Wahl.
best_index = int(np.argmax(silhouette_values))
best_clusters = cluster_counts[best_index]
best_score = silhouette_values[best_index]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Beste getestete Clusterzahl: {best_clusters}")
print(f"Bester Silhouette-Wert: {best_score:.3f}")

# Das Liniendiagramm macht den Vergleich der Clusterzahlen sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_counts, silhouette_values, marker="o", color="tab:blue")
ax.set_title("MiniBatchKMeans: Silhouette nach Clusterzahl")
ax.set_xlabel("Anzahl der Cluster")
ax.set_ylabel("Silhouette-Wert")
ax.set_xticks(cluster_counts)
ax.grid(True, alpha=0.3)
plt.show()



### **1.3. Hierarchisches Clustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_01_03.jpg?v=1784802599" width="250">



>* Keine feste Clusteranzahl nötig
>* Baum zeigt Ähnlichkeitsebenen von Texten

>* Zeigt Cluster auf mehreren Detailstufen
>* Ergebnisse hängen von Distanz und Interpretation ab

>* Anschaulich, aber für große Daten aufwendig
>* Zeigt Beziehungen, jedoch keine Ausreißerwahrscheinlichkeiten



In [ ]:
#@title Python-Code - Hierarchisches Clustering

# Dieses Beispiel zeigt hierarchisches Clustering anschaulich.
# Agglomerative Cluster wachsen schrittweise aus Datenpunkten.
# Die Grafik zeigt Gruppen ohne feste Startzentren.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir erzeugen kleine zweidimensionale Beispieldaten.
features, true_groups = make_blobs(
    n_samples=90,
    centers=3,
    cluster_std=0.75,
    random_state=42,
)

# Skalierung macht Distanzen zwischen Merkmalen vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Eine einfache Prüfung verhindert missverständliche Formen.
if scaled_features.shape != (90, 2):
    raise ValueError("Die Beispieldaten sollten 90 Zeilen und 2 Spalten haben.")

# Agglomeratives Clustering startet mit einzelnen Punkten.
model = AgglomerativeClustering(n_clusters=3, linkage="ward")
cluster_labels = model.fit_predict(scaled_features)

# Wir zählen, wie viele Punkte jedes Cluster enthält.
unique_labels, label_counts = np.unique(cluster_labels, return_counts=True)
count_text = ", ".join(
    [f"Cluster {label}: {count}" for label, count in zip(unique_labels, label_counts)]
)

print(f"scikit-learn Version: {sklearn.__version__}")
print("Hierarchisches Clustering mit Ward-Linkage und drei Zielgruppen.")
print(f"Punkte pro gefundenem Cluster: {count_text}")

# Die Farben zeigen die gefundenen hierarchischen Gruppen.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=55,
)

# Achsen und Titel machen die Darstellung lesbar.
ax.set_title("Hierarchisches Clustering auf skalierten Beispieldaten")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend(*scatter.legend_elements(), title="Cluster")

plt.show()



## **2. Dichte und Anomalien**

### **2.1. Dichtebasierte Cluster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_01.jpg?v=1784802593" width="250">



>* Cluster entstehen in dichten Datenbereichen
>* Erkennt flexible Formen und thematische Nähe

>* Dichteverfahren markieren Rauschen und Ausreißer.
>* Skalierung macht Anomalien sinnvoll interpretierbar.

>* Dichte hängt von Darstellung und Skalierung ab
>* Vorverarbeitung hilft, Anomalien sinnvoll zu erkennen



In [ ]:
#@title Python-Code - Dichtebasierte Cluster

# Wir untersuchen dichtebasierte Cluster und Ausreißer.
# DBSCAN markiert dünn besetzte Punkte als Rauschen.
# Die Grafik zeigt Cluster nach sorgfältiger Skalierung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

# Wir erzeugen zwei gebogene Punktwolken mit Rauschen.
features, true_labels = make_moons(n_samples=260, noise=0.08, random_state=42)

# Einige zusätzliche Punkte simulieren ungewöhnliche Beobachtungen.
outliers = np.array([[-1.6, 1.2], [2.2, -0.7], [2.6, 0.8], [-1.2, -0.8]])
features = np.vstack([features, outliers])

# Diese Prüfung verhindert stille Fehler bei der Datenform.
if features.shape != (264, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Skalierung macht Abstände zwischen Merkmalen vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# DBSCAN sucht dichte Nachbarschaften statt einer festen Clusterzahl.
model = DBSCAN(eps=0.28, min_samples=6)
cluster_labels = model.fit_predict(scaled_features)

# Das Label minus eins bedeutet Rauschen oder mögliche Anomalie.
noise_count = int(np.sum(cluster_labels == -1))
cluster_count = len(set(cluster_labels)) - int(-1 in cluster_labels)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Gefundene Cluster: {cluster_count}")
print(f"Als Rauschen markierte Punkte: {noise_count}")

# Die Farben zeigen Cluster, schwarze Kreuze zeigen Rauschen.
fig, ax = plt.subplots(figsize=(7, 5))
normal_mask = cluster_labels != -1
noise_mask = cluster_labels == -1

scatter = ax.scatter(
    features[normal_mask, 0],
    features[normal_mask, 1],
    c=cluster_labels[normal_mask],
    cmap="viridis",
    s=35,
    label="Dichtebasierte Cluster",
)

ax.scatter(
    features[noise_mask, 0],
    features[noise_mask, 1],
    c="black",
    marker="x",
    s=70,
    label="Rauschen oder Anomalie",
)

ax.set_title("DBSCAN: dichte Cluster und ungewöhnliche Punkte")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



### **2.2. Gaußsche Mischmodelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_02.jpg?v=1784802591" width="250">



>* Weiche Gruppenzuordnung statt harter Clustergrenzen
>* Unsicherheit wird als Wahrscheinlichkeit sichtbar

>* Skalierung verhindert verzerrte Gaußsche Mischmodelle
>* Geringe Modellplausibilität weist auf Anomalien hin

>* Parametrisch für elliptische, überlappende Gruppen
>* Mit Skalierung, Prüfung und Alternativen absichern



In [ ]:
#@title Python-Code - Gaußsche Mischmodelle

# Dieses Beispiel zeigt gaußsche Mischmodelle für Anomalien.
# Skalierung macht Dichten zwischen Merkmalen vergleichbarer.
# Niedrige Modellwahrscheinlichkeit markiert ungewöhnliche Punkte.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Wir erzeugen normale Gruppen und wenige klare Ausreißer.
normal_data, _ = make_blobs(
    n_samples=260,
    centers=[[-2, 0], [2, 1]],
    cluster_std=[0.7, 0.8],
    random_state=42,
)

outliers = np.array([[-5.0, 4.0], [5.0, -3.0], [0.0, 5.0], [6.0, 3.5]])
raw_data = np.vstack([normal_data, outliers])
true_outlier_count = len(outliers)

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if raw_data.shape != (264, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Skalierung verhindert, dass Maßeinheiten die Dichte verzerren.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(raw_data)

# Das Mischmodell lernt zwei weiche, gaußförmige Normalzustände.
gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=42)
gmm.fit(scaled_data)

# Sehr niedrige Log-Wahrscheinlichkeit bedeutet geringe Plausibilität.
log_density = gmm.score_samples(scaled_data)
threshold = np.percentile(log_density, 5)
predicted_outlier = log_density < threshold

# Wir zeigen nur wenige, gut lesbare Kennzahlen.
found_count = int(predicted_outlier.sum())
known_found = int(predicted_outlier[-true_outlier_count:].sum())
print(f"scikit-learn-Version: {sklearn_version}")
print(f"Markierte Anomalien: {found_count} von {len(raw_data)} Punkten")
print(f"Bekannte Ausreißer gefunden: {known_found} von {true_outlier_count}")

# Die Grafik zeigt normale Punkte und niedrige Dichtewerte.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(scaled_data[:, 0], scaled_data[:, 1], c="lightgray", label="normal wirkend")
ax.scatter(
    scaled_data[predicted_outlier, 0],
    scaled_data[predicted_outlier, 1],
    c="crimson",
    label="niedrige Plausibilität",
)

ax.set_title("Gaußsches Mischmodell: Anomalien über geringe Dichte")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend()
plt.show()



### **2.3. Anomalien erkennen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_02_03.jpg?v=1784802595" width="250">



>* Anomalien zeigen ungewöhnliche Datenmuster.
>* Skalierung verhindert verzerrte Modellentscheidungen.

>* Drei Verfahren erkennen Anomalien unterschiedlich.
>* Vergleiche Modelle wegen unterschiedlicher Stärken.

>* Auffälligkeiten immer im Kontext prüfen
>* Unsicherheit dokumentieren, Nachprüfung ermöglichen



In [ ]:
#@title Python-Code - Anomalien erkennen

# Wir erkennen Anomalien in skalierten Beispieldaten.
# IsolationForest markiert ungewöhnliche Punkte als Ausreißer.
# Die Grafik zeigt normale und auffällige Beobachtungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Wir erzeugen normale Punkte und wenige klare Ausreißer.
normal_points, _ = make_blobs(
    n_samples=180,
    centers=[(0, 0)],
    cluster_std=0.8,
    random_state=42,
)

outlier_points = np.array([[4.0, 4.2], [4.5, -3.8], [-4.2, 3.7], [-4.5, -4.0]])
raw_data = np.vstack([normal_points, outlier_points])
true_labels = np.array([0] * len(normal_points) + [1] * len(outlier_points))

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if raw_data.shape != (184, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Skalierung verhindert, dass eine Achse künstlich dominiert.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(raw_data)

# IsolationForest sucht Punkte, die leicht isolierbar sind.
model = IsolationForest(contamination=4 / 184, random_state=42)
prediction = model.fit_predict(scaled_data)

# Scikit-learn nutzt minus eins für erkannte Anomalien.
detected_outliers = prediction == -1
found_count = int(detected_outliers.sum())
correct_count = int(np.sum(detected_outliers & (true_labels == 1)))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Erkannte Anomalien: {found_count} von 184 Punkten")
print(f"Davon absichtlich erzeugte Ausreißer: {correct_count} von 4")

# Die Grafik zeigt die Entscheidung im skalierten Merkmalsraum.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(scaled_data[~detected_outliers, 0], scaled_data[~detected_outliers, 1],
           s=35, label="als normal markiert", alpha=0.75)
ax.scatter(scaled_data[detected_outliers, 0], scaled_data[detected_outliers, 1],
           s=80, label="als Anomalie markiert", marker="x", color="red")
ax.set_title("Anomalieerkennung mit IsolationForest")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend()
plt.show()



## **3. Lernen mit wenigen Labels**

### **3.1. Labels weitergeben**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_01.jpg?v=1784802585" width="250">



>* Wenige sichere Labels, viele unbeschriftete Daten
>* Ähnliche Texte erhalten vorsichtig vorläufige Labels

>* Textrepräsentation bestimmt sinnvolle Nachbarschaften
>* Übertragene Labels bleiben überprüfbare Annahmen

>* Wenige Expertenlabels sparen Annotationsaufwand
>* Vorschläge prüfen und Herkunft dokumentieren



In [ ]:
#@title Python-Code - Labels weitergeben

# Dieses Beispiel zeigt Labelweitergabe mit wenigen sicheren Labels.
# Ähnliche Punkte erhalten vorsichtig vorläufige Klassen.
# Unsicherheit wird über Wahrscheinlichkeiten sichtbar dokumentiert.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.semi_supervised import LabelSpreading

# Wir erzeugen drei gut getrennte Gruppen als einfache Datenwelt.
features, true_labels = make_blobs(
    n_samples=90,
    centers=3,
    cluster_std=1.0,
    random_state=42,
)

# Nur wenige Punkte behalten ihr echtes Label.
known_labels = np.full(true_labels.shape, -1)
for class_id in range(3):
    first_index = np.where(true_labels == class_id)[0][0]
    known_labels[first_index] = class_id

# Diese Prüfung macht die Annahme des Beispiels explizit.
if np.sum(known_labels != -1) != 3:
    raise ValueError("Es sollten genau drei Startlabels vorhanden sein.")

# LabelSpreading verteilt Labels entlang lokaler Ähnlichkeit.
model = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=30)
model.fit(features, known_labels)

# Hohe maximale Wahrscheinlichkeit bedeutet geringere Unsicherheit.
predicted_labels = model.transduction_
confidence = model.label_distributions_.max(axis=1)
uncertain_count = int(np.sum(confidence < 0.8))
accuracy = np.mean(predicted_labels == true_labels)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Sichere Startlabels: {np.sum(known_labels != -1)} von {len(known_labels)}")
print(f"Genauigkeit der weitergegebenen Labels: {accuracy:.2f}")
print(f"Unsichere Vorschläge unter 0.80 Vertrauen: {uncertain_count}")

# Die Grafik trennt sichere Startlabels und weitergegebene Labels.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=predicted_labels,
    cmap="viridis",
    alpha=confidence,
)

# Schwarze Ringe markieren die wenigen menschlich gelabelten Punkte.
known_mask = known_labels != -1
ax.scatter(
    features[known_mask, 0],
    features[known_mask, 1],
    facecolors="none",
    edgecolors="black",
    s=180,
    linewidths=2,
    label="sichere Startlabels",
)

ax.set_title("Labels weitergeben und Unsicherheit dokumentieren")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend(loc="best")
plt.show()



### **3.2. Label Spreading**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_02.jpg?v=1784802587" width="250">



>* Wenige Labels über ähnliche Daten verbreiten
>* Nützlich bei teurer manueller Annotation

>* Labels wandern durch ein Ähnlichkeitsnetzwerk.
>* Gute Merkmale bestimmen sinnvolle Glättung.

>* Unsicherheit bei erzeugten Labels immer dokumentieren
>* Unsichere Fälle gezielt menschlich prüfen



In [ ]:
#@title Python-Code - Label Spreading

# Wir demonstrieren Label Spreading mit wenigen bekannten Labels.
# Unsicherheit wird über Klassenwahrscheinlichkeiten sichtbar gemacht.
# Das Diagramm zeigt sichere und unsichere Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading

# Ein kleiner Datensatz bildet zwei gebogene Klassen.
features, true_labels = make_moons(
    n_samples=180,
    noise=0.12,
    random_state=42,
)

# Skalierung macht Abstände für das Ähnlichkeitsnetz vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Nur wenige Punkte behalten ihr echtes Label.
known_labels = np.full(true_labels.shape, -1)
known_indices = np.array([0, 4, 8, 12, 16, 20, 24, 28])
known_labels[known_indices] = true_labels[known_indices]

# Eine einfache Prüfung verhindert ein irreführendes Beispiel.
if len(np.unique(known_labels[known_labels != -1])) != 2:
    raise ValueError("Die wenigen bekannten Labels müssen beide Klassen enthalten.")

# Label Spreading verteilt Labels über ähnliche Nachbarn.
model = LabelSpreading(kernel="knn", n_neighbors=12, alpha=0.2, max_iter=50)
model.fit(scaled_features, known_labels)

# Wahrscheinlichkeiten zeigen, wie eindeutig die Zuordnung wirkt.
predicted_labels = model.transduction_
probabilities = model.label_distributions_
confidence = probabilities.max(axis=1)

# Unsichere Fälle liegen nahe an einer ausgeglichenen Entscheidung.
unlabeled_mask = known_labels == -1
uncertain_scores = confidence[unlabeled_mask]
uncertain_count = int(np.sum(uncertain_scores < 0.75))
accuracy = np.mean(predicted_labels[unlabeled_mask] == true_labels[unlabeled_mask])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Bekannte Labels: {np.sum(known_labels != -1)} von {len(known_labels)}")
print(f"Genauigkeit auf ursprünglich unbeschrifteten Punkten: {accuracy:.2f}")
print(f"Unsichere automatische Labels unter 0.75 Sicherheit: {uncertain_count}")

# Die Farbe zeigt die Klasse, die Transparenz die Sicherheit.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=predicted_labels,
    cmap="coolwarm",
    alpha=0.25 + 0.75 * confidence,
    s=45,
)

# Schwarze Ringe markieren die wenigen menschlich bekannten Labels.
ax.scatter(
    scaled_features[known_indices, 0],
    scaled_features[known_indices, 1],
    facecolors="none",
    edgecolors="black",
    s=130,
    linewidths=1.8,
    label="bekannte Labels",
)

ax.set_title("Label Spreading: wenige Labels, sichtbare Unsicherheit")
ax.set_xlabel("skaliertes Merkmal 1")
ax.set_ylabel("skaliertes Merkmal 2")
ax.legend(loc="best")
plt.show()



### **3.3. Unsicherheit dokumentieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_12/Lecture_A/image_03_03.jpg?v=1784802589" width="250">



>* Vorhersagen brauchen sichtbare Vertrauensangaben
>* Grenzfälle nicht wie eindeutige Fälle behandeln

>* Unsicherheitsquellen klar benennen
>* Grenzfälle sichtbar machen und prüfen

>* Unsicherheit begrenzt Modellautorität verantwortungsvoll
>* Unsichere Fälle prüfen und gezielt nachlabeln



In [ ]:
#@title Python-Code - Unsicherheit dokumentieren

# Wir dokumentieren Unsicherheit bei wenigen Labels.
# Label Spreading liefert Wahrscheinlichkeiten pro Klasse.
# Unsichere Punkte werden für Prüfung markiert.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler

# Diese Daten simulieren zwei überlappende Gruppen.
features, true_labels = make_blobs(
    n_samples=120,
    centers=[[-1.2, 0.0], [1.2, 0.0]],
    cluster_std=1.15,
    random_state=42,
)

# Nur wenige Beispiele behalten ihr echtes Label.
known_indices = np.array([0, 7, 18, 31, 44, 59, 73, 88])
partial_labels = np.full(true_labels.shape, -1)
partial_labels[known_indices] = true_labels[known_indices]

# Skalierung macht Abstände für das Modell vergleichbarer.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Label Spreading überträgt wenige Labels auf ähnliche Punkte.
model = LabelSpreading(kernel="knn", n_neighbors=10, alpha=0.2)
model.fit(scaled_features, partial_labels)

# Die höchste Klassenwahrscheinlichkeit dient als einfache Sicherheit.
predicted_labels = model.transduction_
confidence = model.label_distributions_.max(axis=1)
uncertainty = 1.0 - confidence

# Wir prüfen, ob alle Punkte eine Unsicherheit erhalten haben.
if len(uncertainty) != len(features):
    raise ValueError("Jeder Datenpunkt braucht einen Unsicherheitswert.")

# Die unsichersten Fälle sind gute Kandidaten für menschliche Prüfung.
review_order = np.argsort(uncertainty)[-5:][::-1]
review_table = pd.DataFrame(
    {
        "Punkt": review_order,
        "Label": predicted_labels[review_order],
        "Sicherheit": np.round(confidence[review_order], 2),
        "Unsicherheit": np.round(uncertainty[review_order], 2),
    }
)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Bekannte Labels: {len(known_indices)} von {len(features)}")
print("Fünf Fälle für menschliche Prüfung:")
print(review_table.to_string(index=False))

# Die Farbe zeigt das Label, die Größe zeigt Unsicherheit.
fig, ax = plt.subplots(figsize=(7, 5))
point_sizes = 40 + 260 * uncertainty
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=predicted_labels,
    s=point_sizes,
    cmap="coolwarm",
    alpha=0.75,
)

# Bekannte Labels werden zusätzlich schwarz umrandet.
ax.scatter(
    scaled_features[known_indices, 0],
    scaled_features[known_indices, 1],
    facecolors="none",
    edgecolors="black",
    s=180,
    linewidths=1.5,
    label="wenige bekannte Labels",
)

ax.set_title("Dokumentierte Unsicherheit bei Label Spreading")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
ax.legend(loc="upper right")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Unüberwacht lernen**</font>


In this lecture, you learned to:
- Vergleichen KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture. 
- Wenden IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierte Daten an. 
- Untersuchen semi-supervised Verfahren mit wenigen Labels und dokumentieren Unsicherheit. 

In the next Lecture (Lecture B), we will go over 'Text als Merkmale'